<!-- course-header -->
# Lesson 03: Machine learning fundamentals

*Companion notebook for lesson 03. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/03-ml-fundamentals.html) (it works well on a phone) or [offline](../lessons/03-ml-fundamentals.html). Worked solutions are in `solutions/03-ml-fundamentals-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/03-ml-fundamentals.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

What it means for a model to learn, the workflow every ML project follows, why models overfit and how to stop them, and how to measure performance honestly.

**Runs on:** Any computer (CPU).

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy scikit-learn

## Splits, overfitting and generalisation

### Three sets, three jobs

> **Split by what you will see in production**
>
> If the model will predict the *future*, split by time: train on the past, validate on later data, never shuffle across time. If it will see *new users* or *new instruments*, make sure no user or instrument appears in both training and test sets. Random splits on such data give scores that look great and mean nothing.

### Overfitting, seen with your own eyes

Fit polynomials of increasing degree to 20 noisy points from a sine wave and compare training and test error.

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(1)
x = rng.uniform(0, 1, 40)
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.25, 40)
X = x.reshape(-1, 1)                                   # sklearn wants 2-D inputs: (samples, features)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)

for degree in [1, 3, 9, 15]:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_tr, y_tr)
    tr = mean_squared_error(y_tr, model.predict(X_tr))
    te = mean_squared_error(y_te, model.predict(X_te))
    print(f"degree {degree:>2}: train MSE {tr:.3f}   test MSE {te:.3f}")

```text
degree  1: train MSE 0.161   test MSE 0.145     underfitting: too simple
degree  3: train MSE 0.027   test MSE 0.037     about right
degree  9: train MSE 0.017   test MSE 0.118     starting to overfit
degree 15: train MSE 0.006   test MSE 95408.7   memorised the noise
```

### Regularisation

L2 (ridge) shrinks all weights smoothly. L1 (lasso) pushes many weights to exactly zero, which also performs feature selection. In neural networks the same L2 idea appears as **weight decay**, alongside dropout and early stopping (lesson 05).

In [ ]:
for alpha in [1e-6, 1e-3, 1e-1, 10]:
    model = make_pipeline(PolynomialFeatures(15), Ridge(alpha=alpha))
    model.fit(X_tr, y_tr)
    print(alpha, round(mean_squared_error(y_te, model.predict(X_te)), 3))
# 1e-06 0.035   even a tiny penalty rescues the degree-15 model
# 0.001 0.031   best
# 0.1   0.083
# 10    0.275   too much: now it underfits

### Cross-validation

With small datasets, one validation split is noisy: you might get lucky or unlucky. **k-fold cross-validation** splits the data into $k$ folds, trains $k$ times, each time validating on a different fold, and averages the scores.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

model = make_pipeline(PolynomialFeatures(3), LinearRegression())
scores = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error")
print(f"MSE {-scores.mean():.3f} ± {scores.std():.3f}")

## Metrics that tell the truth

### Classification and the confusion matrix

Which matters more depends on the cost of each mistake. Screening for a serious disease favours recall; flagging emails as spam favours precision (losing a real email is worse than seeing some spam).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0, 0, 0])
print(confusion_matrix(y_true, y_pred))      # [[5 1] [1 3]]: TN=5 FP=1 FN=1 TP=3
print(classification_report(y_true, y_pred))

## Baselines and leakage

### Data leakage

> **The leakage smell test**
>
> If a result seems too good, it probably is. Ask of every feature: would I really know this value at the moment I need the prediction?

### Hands-on: an honest evaluation

Use scikit-learn's built-in breast cancer dataset (`from sklearn.datasets import load_breast_cancer`).

1. Split 80/20 with `stratify=y` so both sets keep the class balance.
2. Score a `DummyClassifier(strategy="most_frequent")` baseline.
3. Train a `LogisticRegression` inside a pipeline with `StandardScaler`, using 5-fold cross-validation on the training set to pick `C` from `[0.01, 0.1, 1, 10]`.
4. Report precision, recall and F1 on the test set, once.
5. In this problem, which error is worse, a false negative or a false positive? How would you change the threshold?

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/03-ml-fundamentals.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.